# ColorCheck Reference Matcher Training

**Goal:** learn a compact 3D LUT from an unedited target and an edited reference, transferring visual treatment without generating or replacing image content.

This private notebook trains original PyTorch weights on MIT-Adobe FiveK. It first runs a deliberately small end-to-end experiment. Full training remains disabled until the dataset path, GPU, metrics, and checkpoints are confirmed.

## 1. Verify the Kaggle environment

The attached dataset is discovered beneath read-only `/kaggle/input`. Kaggle may add owner and dataset-slug wrapper directories. Checkpoints and metrics are written to `/kaggle/working`, which Kaggle preserves as notebook output when the run completes.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

DATASET_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/colorcheck-reference-matcher')
PACKAGE_ROOT = Path('/kaggle/working/tools/ml/reference_matcher')
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
(PACKAGE_ROOT.parent / '__init__.py').touch()
(PACKAGE_ROOT / '__init__.py').touch()

assert DATASET_ROOT.exists(), f'Dataset was not mounted: {DATASET_ROOT}'
assert torch.cuda.is_available(), 'Enable a GPU accelerator in notebook settings.'
print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'dataset_root': str(DATASET_ROOT),
})


## 2. Materialize the tested training package

Each following cell is generated from the corresponding repository file. The code remains visible for study and review rather than being hidden in a downloaded binary.

### `dataset.py`


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/dataset.py
"""Load Adobe FiveK without leaking the same scene across data splits.

The training code supports a ZIP archive for local experiments and an extracted
directory for Kaggle. Both layouts contain ``raw`` originals and the five expert
edits in directories ``a`` through ``e``.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import zipfile
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF

# Adobe FiveK provides five independent professional edits for each photograph.
EXPERTS = ("a", "b", "c", "d", "e")
REQUIRED_DIRECTORIES = ("raw", *EXPERTS)


def apply_synthetic_style(image: torch.Tensor, seed: int) -> torch.Tensor:
    """Apply one deterministic, photograph-safe global editing recipe.

    The same seed gives two unrelated raw scenes the same treatment. Using one
    transformed scene as the reference and the other as the expected target
    creates exact supervision for arbitrary reference matching. This complements
    FiveK's real expert edits, whose intent is valuable but not parameterized.
    """
    generator = torch.Generator().manual_seed(seed)
    values = torch.rand(10, generator=generator, dtype=image.dtype)
    exposure = 2 ** float(values[0] * 2 - 1)
    contrast = float(0.72 + values[1] * 0.66)
    gamma = float(0.82 + values[2] * 0.38)
    saturation = float(0.65 + values[3] * 0.75)
    channel_gains = 0.86 + values[4:7, None, None] * 0.28
    shadow_lift = float(values[7] * 0.20 - 0.10)
    highlight_lift = float(values[8] * 0.20 - 0.10)
    black_lift = float(values[9] * 0.06 - 0.03)

    styled = image * channel_gains * exposure
    luminance = (
        styled * styled.new_tensor([0.2126, 0.7152, 0.0722])[:, None, None]
    ).sum(0, keepdim=True)
    styled = luminance + (styled - luminance) * saturation
    styled = (styled - 0.5) * contrast + 0.5
    lift = (
        shadow_lift * (1 - luminance.clamp(0, 1)).square()
        + highlight_lift * luminance.clamp(0, 1).square()
        + black_lift * (1 - luminance.clamp(0, 1)).pow(4)
    )
    styled = (styled + lift).clamp(0, 1).pow(gamma)
    return styled.clamp(0, 1)


def synthetic_style_lut(size: int, seed: int) -> torch.Tensor:
    """Materialize the exact 3D LUT implied by one synthetic editing recipe.

    Supervising the complete color cube teaches colors absent from a particular
    training crop and matches the artifact deployed in the browser. Axis order
    is channel, blue, green, red, the same contract used by apply_lut.
    """
    values = torch.linspace(0, 1, size)
    blue, green, red = torch.meshgrid(values, values, values, indexing="ij")
    identity = torch.stack((red, green, blue))
    styled = apply_synthetic_style(identity.reshape(3, size * size, size), seed)
    return styled.reshape(3, size, size, size)


@dataclass(frozen=True)
class SplitManifest:
    """Scene names assigned to training, validation, and final testing.

    Splitting by scene is essential: all six versions of one photograph must
    remain together, or evaluation would contain images already seen in training.
    """

    train: tuple[str, ...]
    validation: tuple[str, ...]
    test: tuple[str, ...]

    def save(self, path: Path) -> None:
        """Save the exact split so later runs evaluate the same photographs."""
        payload = {
            "version": 1,
            "method": "sha256(seed + scene filename), sorted by digest",
            "train": list(self.train),
            "validation": list(self.validation),
            "test": list(self.test),
        }
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2) + "\n")


def _contains_fivek_folders(path: Path) -> bool:
    """Return whether a directory directly contains raw and all expert folders."""
    return path.is_dir() and all((path / name).is_dir() for name in REQUIRED_DIRECTORIES)


def find_dataset_root(path: Path) -> Path:
    """Find the FiveK folder inside a Kaggle dataset mount.

    Kaggle mounts datasets under a slug such as
    ``/kaggle/input/adobe-fivek``. This function also tolerates one or more
    wrapper directories so the notebook does not depend on an accidental path.
    """
    path = Path(path)
    if _contains_fivek_folders(path):
        return path

    candidates: list[Path] = []
    if path.is_dir():
        for raw_directory in path.rglob("raw"):
            if _contains_fivek_folders(raw_directory.parent):
                candidates.append(raw_directory.parent.resolve())

    unique = sorted(set(candidates))
    if len(unique) == 1:
        return unique[0]
    if not unique:
        raise ValueError(
            f"Could not find raw/a/b/c/d/e folders beneath dataset path: {path}"
        )
    raise ValueError(
        "More than one FiveK layout was found. Pass the exact directory with "
        f"--dataset-root. Candidates: {', '.join(map(str, unique))}"
    )


def inspect_dataset(dataset_path: Path) -> tuple[str, ...]:
    """Return scene filenames that have an original and all five expert edits."""
    dataset_path = Path(dataset_path)
    if dataset_path.is_file():
        if not zipfile.is_zipfile(dataset_path):
            raise ValueError(f"Dataset file is not a readable ZIP: {dataset_path}")
        with zipfile.ZipFile(dataset_path) as bundle:
            members = {
                name.replace("\\", "/")
                for name in bundle.namelist()
                if not name.endswith("/")
            }
        source_names = sorted(
            name.removeprefix("raw/") for name in members if name.startswith("raw/")
        )
        complete = [
            name
            for name in source_names
            if all(f"{expert}/{name}" in members for expert in EXPERTS)
        ]
    else:
        root = find_dataset_root(dataset_path)
        source_names = sorted(
            str(file.relative_to(root / "raw")).replace("\\", "/")
            for file in (root / "raw").rglob("*")
            if file.is_file()
        )
        complete = [
            name
            for name in source_names
            if all((root / expert / name).is_file() for expert in EXPERTS)
        ]

    missing_count = len(source_names) - len(complete)
    if missing_count:
        raise ValueError(f"Dataset has {missing_count} source scenes missing expert edits")
    if not complete:
        raise ValueError("No complete raw/A-E scenes found in the dataset")
    return tuple(complete)


def inspect_archive(archive: Path) -> tuple[str, ...]:
    """Keep the original public function name for older local commands."""
    return inspect_dataset(archive)


def create_splits(
    scenes: tuple[str, ...], seed: int = 42, validation_size: int = 500, test_size: int = 500
) -> SplitManifest:
    """Create deterministic, scene-level train/validation/test partitions."""
    if validation_size + test_size >= len(scenes):
        raise ValueError("Validation and test splits leave no training scenes")

    # Hash ranking makes the split independent of filesystem ordering while the
    # seed lets a future experiment deliberately choose a different partition.
    ranked = sorted(
        scenes,
        key=lambda name: hashlib.sha256(f"{seed}:{name}".encode()).digest(),
    )
    test = tuple(ranked[:test_size])
    validation = tuple(ranked[test_size : test_size + validation_size])
    train = tuple(ranked[test_size + validation_size :])
    return SplitManifest(train=train, validation=validation, test=test)


class FiveKReferenceDataset(Dataset):
    """Produce supervised examples for reference-conditioned color matching.

    Every sample contains:

    * ``source``: an unedited photograph that ColorCheck must transform;
    * ``reference``: a different scene edited by one particular expert; and
    * ``expected``: that expert's edit of the source photograph.

    Using a different reference scene prevents the network from copying scene
    content and instead asks it to infer the expert's visual treatment.
    """

    def __init__(
        self,
        dataset_path: Path,
        scenes: tuple[str, ...],
        image_size: int = 256,
        training: bool = False,
        seed: int = 42,
        synthetic_probability: float = 0.0,
        lut_size: int = 33,
        include_lut_supervision: bool = False,
    ) -> None:
        if len(scenes) < 2:
            raise ValueError("At least two scenes are required for reference pairing")
        self.dataset_path = Path(dataset_path)
        self.dataset_root = (
            None if self.dataset_path.is_file() else find_dataset_root(self.dataset_path)
        )
        self.scenes = scenes
        self.image_size = image_size
        self.training = training
        self.seed = seed
        if not 0 <= synthetic_probability <= 1:
            raise ValueError("Synthetic probability must be between zero and one")
        self.synthetic_probability = synthetic_probability
        self.lut_size = lut_size
        self.include_lut_supervision = include_lut_supervision
        # ZIP handles cannot be shared safely by DataLoader worker processes.
        self._bundle: zipfile.ZipFile | None = None

    def __getstate__(self):
        """Remove an open ZIP handle before DataLoader copies this object."""
        state = self.__dict__.copy()
        state["_bundle"] = None
        return state

    def __len__(self) -> int:
        """Expose every scene once for each of the five expert styles."""
        return len(self.scenes) * len(EXPERTS)

    @property
    def bundle(self) -> zipfile.ZipFile:
        """Open the archive lazily inside the process that will read it."""
        if self._bundle is None:
            self._bundle = zipfile.ZipFile(self.dataset_path)
        return self._bundle

    def _prepare_image(self, image: Image.Image) -> torch.Tensor:
        """Convert one image into the consistent tensor format used by PyTorch."""
        image = image.convert("RGB")
        image = TF.resize(
            image,
            [self.image_size, self.image_size],
            interpolation=TF.InterpolationMode.BICUBIC,
            antialias=True,
        )
        # PIL uses height-width-channel integers in [0, 255]. ``to_tensor``
        # returns channel-height-width float values normalized to [0, 1].
        return TF.to_tensor(image)

    @lru_cache(maxsize=384)
    def _read(self, member: str) -> torch.Tensor:
        """Decode and cache a bounded working-size image inside each loader worker.

        FiveK reuses every raw scene across five expert samples and may reuse an
        expert image as another sample's reference. Caching the resized tensor
        avoids repeated JPEG decoding while keeping memory bounded. Returned
        tensors are never modified in place.
        """
        if self.dataset_root is None:
            with self.bundle.open(member) as stream:
                with Image.open(io.BytesIO(stream.read())) as image:
                    return self._prepare_image(image)
        with Image.open(self.dataset_root / member) as image:
            return self._prepare_image(image)

    def __getitem__(self, index: int):
        """Map a flat DataLoader index to a scene, expert, and reference scene."""
        scene_index, expert_index = divmod(index, len(EXPERTS))
        scene = self.scenes[scene_index]
        expert = EXPERTS[expert_index]

        # A stable hash selects a repeatable reference that can never equal the
        # source scene. Repeatability is especially important for validation.
        offset_seed = int.from_bytes(
            hashlib.sha256(f"{self.seed}:{scene}:{expert}".encode()).digest()[:8], "big"
        )
        reference_index = (scene_index + 1 + offset_seed % (len(self.scenes) - 1)) % len(
            self.scenes
        )
        reference_scene = self.scenes[reference_index]

        source = self._read(f"raw/{scene}")
        # Deterministically mix exact synthetic supervision with real edits.
        # Five expert-indexed samples per scene provide five distinct seeds.
        style_seed = int.from_bytes(
            hashlib.sha256(f"style:{self.seed}:{scene}:{expert}".encode()).digest()[:8],
            "big",
        )
        synthetic = (style_seed % 10_000) / 10_000 < self.synthetic_probability
        if synthetic:
            reference = apply_synthetic_style(
                self._read(f"raw/{reference_scene}"), style_seed
            )
            expected = apply_synthetic_style(source, style_seed)
        else:
            reference = self._read(f"{expert}/{reference_scene}")
            expected = self._read(f"{expert}/{scene}")

        # Flip source and expected together. This adds compositional variety
        # without changing the color relationship that the network must learn.
        if self.training and random.random() < 0.5:
            source = source.flip(-1)
            expected = expected.flip(-1)

        sample = {
            "source": source,
            "reference": reference,
            "expected": expected,
            "scene": scene,
            "reference_scene": reference_scene,
            "expert": expert_index,
            "supervision": "synthetic" if synthetic else "fivekExpert",
            # A style identifier supports the training-time wrong-reference
            # control. Synthetic recipes are unique; real edits share an expert.
            "style_id": f"synthetic:{style_seed}" if synthetic else f"expert:{expert}",
        }
        if self.include_lut_supervision:
            sample["expected_lut"] = (
                synthetic_style_lut(self.lut_size, style_seed)
                if synthetic
                else torch.zeros(3, self.lut_size, self.lut_size, self.lut_size)
            )
            sample["has_exact_lut"] = synthetic
        return sample


### `model.py`


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/model.py
"""Small reference-conditioned model that predicts a browser-applicable 3D LUT."""

from __future__ import annotations

import torch
from torch import nn


class ConvBlock(nn.Sequential):
    """Downsample an image while learning increasingly abstract visual features."""

    def __init__(self, incoming: int, outgoing: int) -> None:
        super().__init__(
            # Depthwise convolution examines spatial neighborhoods separately in
            # each channel; the 1x1 convolution then mixes channels efficiently.
            nn.Conv2d(incoming, incoming, 3, stride=2, padding=1, groups=incoming),
            nn.Conv2d(incoming, outgoing, 1),
            nn.GroupNorm(min(8, outgoing), outgoing),
            nn.SiLU(),
        )


class ImageEncoder(nn.Module):
    """Summarize a photograph as a compact 96-number feature vector."""

    def __init__(self, feature_size: int = 96) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, stride=2, padding=2),
            nn.SiLU(),
            ConvBlock(16, 24),
            ConvBlock(24, 40),
            ConvBlock(40, 64),
            ConvBlock(64, feature_size),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        # Adaptive pooling reduces any supported image size to one value per
        # feature channel. ``flatten(1)`` changes BxCx1x1 into BxC.
        return self.features(image).flatten(1)


def identity_lut(size: int) -> torch.Tensor:
    """Create a 3D lookup table that returns every RGB color unchanged."""
    values = torch.linspace(0, 1, size)
    blue, green, red = torch.meshgrid(values, values, values, indexing="ij")
    return torch.stack((red, green, blue))


def initial_lut_bases(size: int, basis_count: int) -> torch.Tensor:
    """Create distinct, paired photographic anchors with a near-neutral mean.

    Identically initialized LUTs make mixture coefficients almost irrelevant and
    starve the reference predictor of useful gradients. These mild warm/cool,
    contrast, saturation, and gamma pairs break that symmetry without imposing
    an extreme look. Every value remains trainable after initialization.
    """
    identity = identity_lut(size)
    luminance = (
        identity * identity.new_tensor([0.2126, 0.7152, 0.0722])[:, None, None, None]
    ).sum(0, keepdim=True)
    anchors = [
        identity * identity.new_tensor([1.06, 1.0, 0.94])[:, None, None, None],
        identity * identity.new_tensor([0.94, 1.0, 1.06])[:, None, None, None],
        (identity - 0.5) * 1.08 + 0.5,
        (identity - 0.5) * 0.92 + 0.5,
        luminance + (identity - luminance) * 1.12,
        luminance + (identity - luminance) * 0.88,
        identity.clamp_min(0).pow(0.92),
        identity.clamp_min(0).pow(1.08),
    ]
    selected = [anchors[index % len(anchors)] for index in range(basis_count)]
    return torch.stack(selected).clamp(0, 1)


class ReferenceLutModel(nn.Module):
    """Predict a safe blend of learned LUTs from target and reference features.

    A 3D lookup table maps an input RGB triplet to a new RGB triplet. The model
    learns several reusable LUT bases, then predicts non-negative coefficients
    that sum to one. This convex blend is bounded and compact enough for browser
    inference, unlike a network that generates every full-resolution pixel.
    """

    def __init__(self, basis_count: int = 8, lut_size: int = 33) -> None:
        super().__init__()
        self.basis_count = basis_count
        self.lut_size = lut_size
        self.encoder = ImageEncoder()
        self.predictor = nn.Sequential(
            nn.Linear(96 * 5 + 12, 160),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(160, basis_count),
        )
        self.basis = nn.Parameter(initial_lut_bases(lut_size, basis_count))

    @staticmethod
    def statistics(image: torch.Tensor) -> torch.Tensor:
        """Describe each RGB channel with its mean and standard deviation."""
        mean = image.mean(dim=(2, 3))
        std = image.std(dim=(2, 3), correction=0)
        return torch.cat((mean, std), dim=1)

    def predict_coefficients(self, source: torch.Tensor, reference: torch.Tensor):
        """Decide how strongly each learned LUT basis should influence the result."""
        # One batched encoder call produces a smaller ONNX graph and reduces
        # browser runtime dispatch overhead compared with two duplicated calls.
        paired_features = self.encoder(torch.cat((source, reference), dim=0))
        source_features, reference_features = paired_features.chunk(2, dim=0)
        # The model sees each image, their feature difference, and simple color
        # statistics. Their combination answers both "what is present?" and
        # "how does the desired reference differ from this source?"
        features = torch.cat(
            (
                source_features,
                reference_features,
                reference_features - source_features,
                (reference_features - source_features).abs(),
                reference_features * source_features,
                self.statistics(source),
                self.statistics(reference),
            ),
            dim=1,
        )
        scores = self.predictor(features)
        # This is a numerically stable softmax written as primitive operations.
        # ONNX Runtime 1.19's static quantizer cannot calibrate a Softmax node,
        # while ReduceMax/Exp/ReduceSum/Div preserve the same smooth convex
        # weights in PyTorch, native ONNX, and ONNX Runtime Web.
        shifted = scores - scores.amax(dim=1, keepdim=True)
        exponentials = shifted.exp()
        return exponentials / exponentials.sum(dim=1, keepdim=True)

    def forward(self, source: torch.Tensor, reference: torch.Tensor):
        coefficients = self.predict_coefficients(source, reference)
        bounded_basis = self.basis.clamp(0, 1)
        lut = torch.einsum("bk,kcxyz->bcxyz", coefficients, bounded_basis)
        return lut, coefficients


def apply_lut(image: torch.Tensor, lut: torch.Tensor) -> torch.Tensor:
    """Apply a 3D LUT with differentiable trilinear interpolation.

    An RGB color usually falls between LUT grid points. The eight surrounding
    colors are sampled and blended according to distance. Because PyTorch can
    differentiate through these blends, training can update both the LUT bases
    and the network that chooses their coefficients.
    """
    batch, channels, height, width = image.shape
    size = lut.shape[-1]
    scaled = image.clamp(0, 1) * (size - 1)
    lower = scaled.floor().long()
    upper = (lower + 1).clamp(max=size - 1)
    # Autocast may produce a BF16/FP16 LUT while the source image remains FP32.
    # ``lerp`` requires its interpolation weights to match the sampled LUT dtype.
    fraction = (scaled - lower).to(dtype=lut.dtype)
    red0, green0, blue0 = lower[:, 0], lower[:, 1], lower[:, 2]
    red1, green1, blue1 = upper[:, 0], upper[:, 1], upper[:, 2]
    red_weight, green_weight, blue_weight = fraction[:, 0], fraction[:, 1], fraction[:, 2]
    flattened = lut.reshape(batch, channels, -1)

    def sample(red, green, blue):
        # Flattening converts a 3D coordinate into one gather index. This avoids
        # a Python loop over pixels and keeps interpolation accelerator-friendly.
        index = (blue * size * size + green * size + red).reshape(batch, 1, -1)
        return flattened.gather(2, index.expand(-1, channels, -1)).reshape(
            batch, channels, height, width
        )

    c000 = sample(red0, green0, blue0)
    c100 = sample(red1, green0, blue0)
    c010 = sample(red0, green1, blue0)
    c110 = sample(red1, green1, blue0)
    c001 = sample(red0, green0, blue1)
    c101 = sample(red1, green0, blue1)
    c011 = sample(red0, green1, blue1)
    c111 = sample(red1, green1, blue1)
    rw, gw, bw = red_weight[:, None], green_weight[:, None], blue_weight[:, None]
    c00 = c000.lerp(c100, rw)
    c10 = c010.lerp(c110, rw)
    c01 = c001.lerp(c101, rw)
    c11 = c011.lerp(c111, rw)
    return c00.lerp(c10, gw).lerp(c01.lerp(c11, gw), bw)


### `losses.py`


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/losses.py
"""Training objectives for faithful color transfer with structural guardrails."""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn


def rgb_to_lab(image: torch.Tensor) -> torch.Tensor:
    """Convert display RGB into perceptual CIE Lab values.

    Equal RGB errors do not always look equally different to a person. Lab is
    used here because Euclidean distance in this space is a more meaningful
    color-error signal than raw channel distance.
    """
    # Undo the nonlinear sRGB transfer curve before converting color spaces.
    linear = torch.where(
        image <= 0.04045,
        image / 12.92,
        ((image + 0.055) / 1.055).pow(2.4),
    )
    matrix = image.new_tensor(
        [[0.4124564, 0.3575761, 0.1804375],
         [0.2126729, 0.7151522, 0.0721750],
         [0.0193339, 0.1191920, 0.9503041]]
    )
    xyz = torch.einsum("ij,bjhw->bihw", matrix, linear)
    xyz = xyz / image.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    delta = 6 / 29
    f = torch.where(xyz > delta**3, xyz.clamp_min(1e-8).pow(1 / 3), xyz / (3 * delta**2) + 4 / 29)
    lightness = 116 * f[:, 1] - 16
    a = 500 * (f[:, 0] - f[:, 1])
    b = 200 * (f[:, 1] - f[:, 2])
    return torch.stack((lightness, a, b), dim=1)


def image_gradients(image: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Measure horizontal and vertical luminance edges in a batch of images."""
    luminance = (image * image.new_tensor([0.2126, 0.7152, 0.0722])[None, :, None, None]).sum(1, keepdim=True)
    return luminance[:, :, :, 1:] - luminance[:, :, :, :-1], luminance[:, :, 1:, :] - luminance[:, :, :-1, :]


def reference_ranking_loss(
    correct_output: torch.Tensor,
    wrong_output: torch.Tensor,
    expected: torch.Tensor,
    margin: float = 0.01,
) -> torch.Tensor:
    """Require the correct reference to beat a known-wrong style per image.

    Reconstruction alone can reward one broadly attractive enhancement. This
    pairwise hinge objective instead asks for the expected edit to be at least
    margin closer with its true reference. It is used only during training and
    adds no operation or model parameter to browser inference.
    """
    correct_error = (correct_output - expected).abs().mean(dim=(1, 2, 3))
    wrong_error = (wrong_output - expected).abs().mean(dim=(1, 2, 3))
    return (margin + correct_error - wrong_error).relu().mean()


def exact_lut_supervision_loss(
    predicted_lut: torch.Tensor,
    expected_lut: torch.Tensor,
    available: torch.Tensor,
) -> torch.Tensor:
    """Measure full-cube error only where an exact synthetic LUT is known."""
    per_sample = (predicted_lut.float() - expected_lut.float()).abs().flatten(1).mean(1)
    weights = available.to(device=per_sample.device, dtype=per_sample.dtype)
    return (per_sample * weights).sum() / weights.sum().clamp_min(1)


class ReferenceMatchLoss(nn.Module):
    """Combine visual accuracy with guardrails against destructive LUTs.

    No single number captures a good correction. Reconstruction and Lab error
    reward agreement with the expert edit; structure protects source edges;
    smoothness discourages color discontinuities; and monotonicity discourages
    inversions where a brighter input unexpectedly becomes darker.
    """
    def __init__(
        self,
        color_weight: float = 0.02,
        structure_weight: float = 2.0,
        smoothness_weight: float = 0.0005,
        monotonicity_weight: float = 0.002,
    ) -> None:
        super().__init__()
        self.color_weight = color_weight
        self.structure_weight = structure_weight
        self.smoothness_weight = smoothness_weight
        self.monotonicity_weight = monotonicity_weight

    def forward(self, output, expected, source, lut):
        # Charbonnier reconstruction behaves like L1 error while remaining
        # smooth around zero, which makes gradient-based optimization stable.
        reconstruction = torch.sqrt((output - expected).square() + 1e-6).mean()
        color = (rgb_to_lab(output) - rgb_to_lab(expected)).square().sum(1).add(1e-6).sqrt().mean()
        output_x, output_y = image_gradients(output)
        source_x, source_y = image_gradients(source)
        structure = F.l1_loss(output_x, source_x) + F.l1_loss(output_y, source_y)
        differences = [
            lut[:, :, 1:] - lut[:, :, :-1],
            lut[:, :, :, 1:] - lut[:, :, :, :-1],
            lut[:, :, :, :, 1:] - lut[:, :, :, :, :-1],
        ]
        smoothness = sum(diff.square().mean() for diff in differences)
        monotonicity = sum((-diff).relu().mean() for diff in differences)
        total = (
            reconstruction
            + self.color_weight * color
            + self.structure_weight * structure
            + self.smoothness_weight * smoothness
            + self.monotonicity_weight * monotonicity
        )
        return total, {
            "reconstruction": reconstruction.detach(),
            "delta_e76": color.detach(),
            "structure": structure.detach(),
            "lut_smoothness": smoothness.detach(),
            "lut_monotonicity": monotonicity.detach(),
        }


### `evaluation.py`


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/evaluation.py
"""Held-out evaluation for reference-conditioned photographic matching.

Training loss answers whether gradient descent can improve the model. This
module answers the harder product question: does the trained model improve an
unseen photograph, use the supplied reference, preserve structure, and remain
small and fast enough for an interactive browser editor?

Every metric is retained per photograph before aggregation. Confidence intervals
resample source scenes, not individual expert variants or pixels, because the five
edits of one FiveK photograph are related observations rather than five independent
photographs.
"""

from __future__ import annotations

import math
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F

from .dataset import EXPERTS
from .losses import image_gradients, rgb_to_lab

# These thresholds are declared in source before full training. They prevent a
# large evaluation set from promoting a statistically positive but visually tiny
# result. Confidence intervals must still exclude zero independently.
MINIMUM_PRACTICAL_EFFECTS = {
    "identityDeltaE2000": 0.25,
    "identitySsim": 0.002,
    "statisticsDeltaE2000": 0.10,
    "correctReferenceDeltaE2000": 0.05,
    "referenceSensitivityMae": 1 / 1024,
}


def _per_image_mean(values: torch.Tensor) -> torch.Tensor:
    """Average every dimension except the leading batch dimension."""
    return values.flatten(1).mean(1)


def _luminance(image: torch.Tensor) -> torch.Tensor:
    """Convert sRGB channels to a one-channel Rec. 709 luminance estimate."""
    weights = image.new_tensor([0.2126, 0.7152, 0.0722])[None, :, None, None]
    return (image * weights).sum(1, keepdim=True)


def ciede2000(lab_a: torch.Tensor, lab_b: torch.Tensor) -> torch.Tensor:
    """Return the CIEDE2000 color difference for corresponding Lab pixels.

    CIEDE2000 accounts for nonuniform sensitivity to lightness, chroma, and hue.
    It is used only as an evaluation metric: the smoother Delta E 76 objective
    remains preferable for backpropagation.
    """
    l1, a1, b1 = lab_a.unbind(1)
    l2, a2, b2 = lab_b.unbind(1)
    c1 = torch.sqrt(a1.square() + b1.square() + 1e-12)
    c2 = torch.sqrt(a2.square() + b2.square() + 1e-12)
    mean_c = (c1 + c2) / 2
    mean_c7 = mean_c.pow(7)
    g = 0.5 * (1 - torch.sqrt(mean_c7 / (mean_c7 + 25**7)))
    a1_prime = (1 + g) * a1
    a2_prime = (1 + g) * a2
    c1_prime = torch.sqrt(a1_prime.square() + b1.square() + 1e-12)
    c2_prime = torch.sqrt(a2_prime.square() + b2.square() + 1e-12)
    h1_prime = torch.rad2deg(torch.atan2(b1, a1_prime)).remainder(360)
    h2_prime = torch.rad2deg(torch.atan2(b2, a2_prime)).remainder(360)

    delta_l = l2 - l1
    delta_c = c2_prime - c1_prime
    delta_h_degrees = h2_prime - h1_prime
    delta_h_degrees = torch.where(
        delta_h_degrees > 180, delta_h_degrees - 360, delta_h_degrees
    )
    delta_h_degrees = torch.where(
        delta_h_degrees < -180, delta_h_degrees + 360, delta_h_degrees
    )
    delta_h_degrees = torch.where(
        (c1_prime * c2_prime) == 0, torch.zeros_like(delta_h_degrees), delta_h_degrees
    )
    delta_h = 2 * torch.sqrt(c1_prime * c2_prime) * torch.sin(
        torch.deg2rad(delta_h_degrees) / 2
    )

    mean_l = (l1 + l2) / 2
    mean_c_prime = (c1_prime + c2_prime) / 2
    hue_distance = (h1_prime - h2_prime).abs()
    mean_h = (h1_prime + h2_prime) / 2
    mean_h = torch.where(hue_distance > 180, mean_h + 180, mean_h).remainder(360)
    mean_h = torch.where(
        (c1_prime * c2_prime) == 0, h1_prime + h2_prime, mean_h
    )
    t = (
        1
        - 0.17 * torch.cos(torch.deg2rad(mean_h - 30))
        + 0.24 * torch.cos(torch.deg2rad(2 * mean_h))
        + 0.32 * torch.cos(torch.deg2rad(3 * mean_h + 6))
        - 0.20 * torch.cos(torch.deg2rad(4 * mean_h - 63))
    )
    delta_theta = 30 * torch.exp(-((mean_h - 275) / 25).square())
    r_c = 2 * torch.sqrt(mean_c_prime.pow(7) / (mean_c_prime.pow(7) + 25**7))
    s_l = 1 + (0.015 * (mean_l - 50).square()) / torch.sqrt(
        20 + (mean_l - 50).square()
    )
    s_c = 1 + 0.045 * mean_c_prime
    s_h = 1 + 0.015 * mean_c_prime * t
    r_t = -torch.sin(torch.deg2rad(2 * delta_theta)) * r_c
    lightness = delta_l / s_l
    chroma = delta_c / s_c
    hue = delta_h / s_h
    return torch.sqrt(
        (lightness.square() + chroma.square() + hue.square() + r_t * chroma * hue)
        .clamp_min(0)
        + 1e-12
    )


def _ssim_per_image(output: torch.Tensor, expected: torch.Tensor) -> torch.Tensor:
    """Measure local luminance, contrast, and structure similarity."""
    x, y = _luminance(output), _luminance(expected)
    kernel = min(11, x.shape[-2], x.shape[-1])
    if kernel % 2 == 0:
        kernel -= 1
    padding = kernel // 2
    mean_x = F.avg_pool2d(x, kernel, stride=1, padding=padding)
    mean_y = F.avg_pool2d(y, kernel, stride=1, padding=padding)
    variance_x = F.avg_pool2d(x.square(), kernel, 1, padding) - mean_x.square()
    variance_y = F.avg_pool2d(y.square(), kernel, 1, padding) - mean_y.square()
    covariance = F.avg_pool2d(x * y, kernel, 1, padding) - mean_x * mean_y
    c1, c2 = 0.01**2, 0.03**2
    score = ((2 * mean_x * mean_y + c1) * (2 * covariance + c2)) / (
        (mean_x.square() + mean_y.square() + c1)
        * (variance_x + variance_y + c2)
    )
    return _per_image_mean(score)


def metric_samples(output: torch.Tensor, expected: torch.Tensor) -> dict[str, torch.Tensor]:
    """Calculate one value per photograph for each complementary quality axis."""
    output, expected = output.float(), expected.float()
    error = output - expected
    mse = _per_image_mean(error.square())
    lab_output, lab_expected = rgb_to_lab(output), rgb_to_lab(expected)
    output_x, output_y = image_gradients(output)
    expected_x, expected_y = image_gradients(expected)
    edge_error = (
        _per_image_mean((output_x - expected_x).abs())
        + _per_image_mean((output_y - expected_y).abs())
    ) / 2
    chroma_output = torch.sqrt(lab_output[:, 1].square() + lab_output[:, 2].square())
    chroma_expected = torch.sqrt(lab_expected[:, 1].square() + lab_expected[:, 2].square())
    return {
        "mae": _per_image_mean(error.abs()),
        "psnrDb": -10 * torch.log10(mse.clamp_min(1e-10)),
        "ssim": _ssim_per_image(output, expected),
        "deltaE2000": _per_image_mean(ciede2000(lab_output, lab_expected)),
        "luminanceMae": _per_image_mean((_luminance(output) - _luminance(expected)).abs()),
        "chromaMae": _per_image_mean((chroma_output - chroma_expected).abs()),
        "edgeMae": edge_error,
        "clippingPercent": _per_image_mean(
            ((output <= 1 / 255) | (output >= 254 / 255))
            .any(dim=1, keepdim=True)
            .float()
        )
        * 100,
    }


def statistics_match(source: torch.Tensor, reference: torch.Tensor) -> torch.Tensor:
    """Apply the editor's bounded per-channel mean/spread reference baseline."""
    source_mean = source.mean((2, 3), keepdim=True)
    source_std = source.std((2, 3), correction=0, keepdim=True)
    reference_mean = reference.mean((2, 3), keepdim=True)
    reference_std = reference.std((2, 3), correction=0, keepdim=True)
    gain = (reference_std / source_std.clamp_min(0.015)).clamp(0.65, 1.5)
    return ((source - source_mean) * gain + reference_mean).clamp(0, 1)


def mismatch_references(
    reference: torch.Tensor, style_labels
) -> tuple[torch.Tensor, torch.Tensor]:
    """Pair each item with a reference carrying a different style label.

    Rolling a batch can accidentally compare two references from the same FiveK
    expert. That is not a meaningful wrong-style control. This deterministic
    mapping searches forward for a different expert and returns the selected
    indices so the evaluation remains auditable.
    """
    count = reference.shape[0]
    indices = torch.arange(count, device=reference.device)
    if count < 2:
        return reference, indices
    if isinstance(style_labels, torch.Tensor):
        label_values = style_labels.detach().cpu().tolist()
    else:
        label_values = list(style_labels)
    if len(label_values) != count:
        raise ValueError("One style label is required for every reference")
    selected = []
    for index in range(count):
        candidate = next(
            (
                (index + offset) % count
                for offset in range(1, count)
                if label_values[(index + offset) % count] != label_values[index]
            ),
            index,
        )
        selected.append(candidate)
    indices = torch.tensor(selected, device=reference.device)
    return reference.index_select(0, indices), indices


def _bootstrap(values: np.ndarray, seed: int, samples: int = 1000) -> dict[str, float]:
    """Estimate a photograph-level 95% confidence interval for a mean."""
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        raise ValueError("Cannot summarize an empty metric")
    if len(values) == 1:
        lower = upper = float(values[0])
    else:
        generator = np.random.default_rng(seed)
        indices = generator.integers(0, len(values), size=(samples, len(values)))
        means = values[indices].mean(1)
        lower, upper = np.percentile(means, [2.5, 97.5])
    return {
        "mean": float(values.mean()),
        "ci95Lower": float(lower),
        "ci95Upper": float(upper),
        "sampleCount": int(len(values)),
    }


def _cluster_bootstrap(
    values: np.ndarray, scene_ids, seed: int, samples: int = 1000
) -> dict[str, float]:
    """Bootstrap scene means so repeated expert edits do not inflate certainty.

    FiveK contains five expert variants of each source scene. Averaging those
    variants within a scene and then resampling scenes gives each photograph one
    independent vote while retaining every expert measurement in its scene mean.
    """
    values = np.asarray(values, dtype=np.float64)
    scenes = np.asarray(scene_ids, dtype=object)
    if len(values) != len(scenes):
        raise ValueError("One source scene identifier is required per metric value")
    unique_scenes = tuple(dict.fromkeys(scenes.tolist()))
    scene_means = np.asarray(
        [values[scenes == scene].mean() for scene in unique_scenes], dtype=np.float64
    )
    result = _bootstrap(scene_means, seed, samples)
    result["sceneCount"] = result.pop("sampleCount")
    result["observationCount"] = int(len(values))
    return result


class MetricStore:
    """Collect per-image measurements while retaining expert identities."""

    def __init__(self) -> None:
        self.values: dict[str, list[float]] = defaultdict(list)
        self.experts: list[int] = []
        self.scenes: list[str] = []

    def add(
        self, metrics: dict[str, torch.Tensor], experts: torch.Tensor, scenes
    ) -> None:
        scene_values = list(scenes)
        batch_size = len(experts)
        if len(scene_values) != batch_size:
            raise ValueError("One source scene identifier is required per photograph")
        for name, values in metrics.items():
            if len(values) != batch_size:
                raise ValueError("Every metric must contain one value per photograph")
            self.values[name].extend(values.detach().cpu().tolist())
        self.experts.extend(experts.detach().cpu().tolist())
        self.scenes.extend(scene_values)

    def summary(self, seed: int = 42) -> dict:
        overall = {
            name: _cluster_bootstrap(np.asarray(values), self.scenes, seed + index)
            for index, (name, values) in enumerate(sorted(self.values.items()))
        }
        expert_array = np.asarray(self.experts)
        scene_array = np.asarray(self.scenes, dtype=object)
        by_expert = {}
        for expert_index, expert_name in enumerate(EXPERTS):
            selected = expert_array == expert_index
            if selected.any():
                by_expert[expert_name.upper()] = {
                    name: _cluster_bootstrap(
                        np.asarray(values)[selected],
                        scene_array[selected],
                        seed + expert_index,
                    )
                    for name, values in sorted(self.values.items())
                }
        return {"overall": overall, "byExpert": by_expert}


def paired_improvement(
    model: MetricStore, baseline: MetricStore, seed: int = 42
) -> dict[str, dict[str, float]]:
    """Report positive values when the model improves over a paired baseline."""
    higher_is_better = {"psnrDb", "ssim"}
    if model.scenes != baseline.scenes:
        raise ValueError("Paired comparisons require metrics in the same scene order")
    comparisons = {}
    for index, name in enumerate(sorted(model.values)):
        model_values = np.asarray(model.values[name])
        baseline_values = np.asarray(baseline.values[name])
        difference = (
            model_values - baseline_values
            if name in higher_is_better
            else baseline_values - model_values
        )
        comparisons[name] = _cluster_bootstrap(
            difference, model.scenes, seed + index
        )
    return comparisons


def coefficient_diagnostics(coefficients: torch.Tensor) -> dict[str, object]:
    """Detect a collapsed mixture that ignores most learned LUT bases."""
    coefficients = coefficients.float()
    entropy = -(coefficients * coefficients.clamp_min(1e-8).log()).sum(1)
    entropy /= math.log(coefficients.shape[1])
    dominant_counts = torch.bincount(
        coefficients.argmax(1), minlength=coefficients.shape[1]
    )
    return {
        "normalizedEntropyMean": float(entropy.mean()),
        "normalizedEntropyMinimum": float(entropy.min()),
        "meanBasisWeights": coefficients.mean(0).tolist(),
        "basisWeightStandardDeviations": coefficients.std(0, correction=0).tolist(),
        "dominantBasisCounts": dominant_counts.tolist(),
        "dominantBasisFraction": float(dominant_counts.max() / coefficients.shape[0]),
    }


def deployment_evidence_gates(
    identity_gain: dict,
    statistics_gain: dict,
    wrong_reference_gain: dict,
    reference_sensitivity_mean: float,
    coefficient_report: dict,
) -> dict[str, bool]:
    """Require statistical confidence, practical effect, and model diversity."""
    minimum = MINIMUM_PRACTICAL_EFFECTS
    return {
        "beatsIdentityDeltaE2000": (
            identity_gain["deltaE2000"]["ci95Lower"] > 0
            and identity_gain["deltaE2000"]["mean"]
            >= minimum["identityDeltaE2000"]
        ),
        "beatsIdentitySsim": (
            identity_gain["ssim"]["ci95Lower"] > 0
            and identity_gain["ssim"]["mean"] >= minimum["identitySsim"]
        ),
        "beatsStatisticsDeltaE2000": (
            statistics_gain["deltaE2000"]["ci95Lower"] > 0
            and statistics_gain["deltaE2000"]["mean"]
            >= minimum["statisticsDeltaE2000"]
        ),
        "correctReferenceBeatsShuffledDeltaE2000": (
            wrong_reference_gain["deltaE2000"]["ci95Lower"] > 0
            and wrong_reference_gain["deltaE2000"]["mean"]
            >= minimum["correctReferenceDeltaE2000"]
        ),
        "respondsToReference": (
            reference_sensitivity_mean >= minimum["referenceSensitivityMae"]
        ),
        "avoidsSingleBasisCollapse": (
            max(coefficient_report["meanBasisWeights"]) < 0.9
            and coefficient_report["dominantBasisFraction"] < 0.9
        ),
        "doesNotIncreaseClippingVsIdentity": (
            identity_gain["clippingPercent"]["ci95Lower"] >= 0
        ),
    }


### `train.py`


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/train.py
"""Train, validate, and test ColorCheck's reference-conditioned 3D LUT model.

This module contains the complete learning loop. The command line interface is
shared by local development and Kaggle so the same model is evaluated under the
same rules in both environments.
"""

from __future__ import annotations

import argparse
import json
import random
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from .dataset import FiveKReferenceDataset, create_splits, inspect_dataset
from .evaluation import (
    MINIMUM_PRACTICAL_EFFECTS,
    MetricStore,
    coefficient_diagnostics,
    deployment_evidence_gates,
    metric_samples,
    mismatch_references,
    paired_improvement,
    statistics_match,
)
from .losses import (
    ReferenceMatchLoss,
    exact_lut_supervision_loss,
    reference_ranking_loss,
    rgb_to_lab,
)
from .model import ReferenceLutModel, apply_lut


def choose_device(requested: str) -> torch.device:
    """Select CUDA on Kaggle, Apple Metal locally, or CPU as a final fallback."""
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def choose_mixed_precision(requested: str, device: torch.device) -> torch.dtype | None:
    """Choose a lower-precision CUDA format without changing stored checkpoints.

    Mixed precision accelerates temporary matrix operations during training.
    Model parameters and saved checkpoints remain full precision. Automatic mode
    deliberately stays off for CPU and Apple Metal, where this pipeline has not
    demonstrated a reliable speed advantage.
    """
    if requested == "off":
        return None
    if requested == "fp16":
        if device.type != "cuda":
            raise ValueError("FP16 training is only enabled for CUDA in this pipeline")
        return torch.float16
    if requested == "bf16":
        if device.type != "cuda" or not torch.cuda.is_bf16_supported():
            raise ValueError("BF16 was requested, but this CUDA device does not support it")
        return torch.bfloat16
    if device.type == "cuda":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return None


def autocast_context(device: torch.device, dtype: torch.dtype | None):
    """Return a PyTorch precision context or a no-op context when AMP is disabled."""
    if dtype is None:
        return nullcontext()
    return torch.autocast(device_type=device.type, dtype=dtype)


def move_images(batch, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transfer only image tensors to the accelerator.

    Metadata such as scene filenames remains on the CPU. ``non_blocking`` lets
    CUDA overlap a transfer with computation when DataLoader uses pinned memory.
    """
    non_blocking = device.type == "cuda"
    return (
        batch["source"].to(device, non_blocking=non_blocking),
        batch["reference"].to(device, non_blocking=non_blocking),
        batch["expected"].to(device, non_blocking=non_blocking),
    )


def batch_metrics(output: torch.Tensor, expected: torch.Tensor) -> dict[str, float]:
    """Calculate interpretable image-quality measurements for one batch."""
    # Metrics are intentionally computed in float32 so mixed precision cannot
    # make the reported validation result appear better or worse by rounding.
    output = output.float()
    expected = expected.float()
    mse = (output - expected).square().mean()
    psnr = -10 * torch.log10(mse.clamp_min(1e-10))
    delta_e = (rgb_to_lab(output) - rgb_to_lab(expected)).square().sum(1).sqrt().mean()
    clipping = (
        ((output <= 1 / 255) | (output >= 254 / 255)).any(dim=1).float().mean() * 100
    )
    return {
        "psnr": psnr.item(),
        "delta_e76": delta_e.item(),
        "clipping_percent": clipping.item(),
    }


@torch.no_grad()
def validate(
    model,
    loader,
    device,
    loss_function,
    mixed_precision: torch.dtype | None = None,
    shuffled_reference: bool = False,
):
    """Evaluate without updating weights and average metrics per photograph."""
    model.eval()
    totals: dict[str, float] = {}
    count = 0
    for batch in tqdm(loader, desc="validate", leave=False):
        source, reference, expected = move_images(batch, device)

        # Rolling references by one position intentionally gives every source
        # the wrong reference. A useful reference-conditioned model should score
        # worse here than it does with the correct references.
        if shuffled_reference and reference.shape[0] > 1:
            reference = reference.roll(1, 0)

        with autocast_context(device, mixed_precision):
            lut, _ = model(source, reference)
            output = apply_lut(source, lut)

        # Keep the loss calculation numerically stable and comparable across
        # GPUs by converting model outputs back to float32.
        loss, components = loss_function(
            output.float(), expected.float(), source.float(), lut.float()
        )
        values = {"loss": loss.item(), **batch_metrics(output, expected)}
        values.update({name: value.item() for name, value in components.items()})
        batch_size = source.shape[0]
        for name, value in values.items():
            totals[name] = totals.get(name, 0.0) + value * batch_size
        count += batch_size
    return {name: value / count for name, value in totals.items()}


@torch.no_grad()
def identity_baseline(loader, device):
    """Measure the result of doing nothing to establish a minimum useful score."""
    totals: dict[str, float] = {}
    count = 0
    for batch in tqdm(loader, desc="identity baseline", leave=False):
        source = batch["source"].to(device, non_blocking=device.type == "cuda")
        expected = batch["expected"].to(device, non_blocking=device.type == "cuda")
        values = batch_metrics(source, expected)
        for name, value in values.items():
            totals[name] = totals.get(name, 0.0) + value * source.shape[0]
        count += source.shape[0]
    return {name: value / count for name, value in totals.items()}


def checkpoint_selection_score(
    real_validation: dict[str, float],
    synthetic_validation: dict[str, float],
) -> float:
    """Balance photographic edits and exact reference transfer during selection.

    Both validation sets use the same held-out scene identities, but ask different
    questions. Real FiveK pairs reward plausible expert enhancement; synthetic
    pairs verify that one style applied to two unrelated photographs can be
    recovered from the reference. Their equally weighted DeltaE76 mean is easy to
    audit and prevents either objective from selecting the checkpoint alone.
    """
    return (real_validation["delta_e76"] + synthetic_validation["delta_e76"]) / 2


@torch.no_grad()
def comprehensive_evaluation(model, loader, device, mixed_precision, seed: int = 42):
    """Run final product-oriented controls after checkpoint selection.

    This deliberately runs only on the untouched test split. It compares the
    learned model with doing nothing and with the editor's current statistics
    matcher, then repeats inference with incorrect references. Results retain
    per-photo uncertainty and expert-specific behavior.
    """
    model.eval()
    stores = {
        "model": MetricStore(),
        "identity": MetricStore(),
        "statisticsMatch": MetricStore(),
        "shuffledReference": MetricStore(),
    }
    sensitivities: list[float] = []
    all_coefficients = []
    elapsed_seconds = 0.0
    peak_memory = 0

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for batch in tqdm(loader, desc="comprehensive evaluation", leave=False):
        source, reference, expected = move_images(batch, device)
        experts = batch["expert"]
        scenes = batch["scene"]
        wrong_reference, _ = mismatch_references(reference, experts)

        if device.type == "cuda":
            torch.cuda.synchronize(device)
        started = time.perf_counter()
        with autocast_context(device, mixed_precision):
            lut, coefficients = model(source, reference)
            output = apply_lut(source, lut)
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        elapsed_seconds += time.perf_counter() - started

        # Wrong-reference inference is an evaluation control, not product
        # latency, so it is deliberately outside the timed region.
        with autocast_context(device, mixed_precision):
            wrong_lut, _ = model(source, wrong_reference)
            wrong_output = apply_lut(source, wrong_lut)

        output = output.float()
        wrong_output = wrong_output.float()
        stores["model"].add(metric_samples(output, expected), experts, scenes)
        stores["identity"].add(metric_samples(source, expected), experts, scenes)
        stores["statisticsMatch"].add(
            metric_samples(statistics_match(source, reference), expected), experts, scenes
        )
        stores["shuffledReference"].add(
            metric_samples(wrong_output, expected), experts, scenes
        )
        sensitivities.extend(
            (output - wrong_output).abs().flatten(1).mean(1).cpu().tolist()
        )
        all_coefficients.append(coefficients.detach().float().cpu())

    if device.type == "cuda":
        peak_memory = torch.cuda.max_memory_allocated(device)

    summaries = {name: store.summary(seed) for name, store in stores.items()}
    identity_gain = paired_improvement(stores["model"], stores["identity"], seed)
    statistics_gain = paired_improvement(
        stores["model"], stores["statisticsMatch"], seed
    )
    wrong_reference_gain = paired_improvement(
        stores["model"], stores["shuffledReference"], seed
    )
    sensitivity = np.asarray(sensitivities)
    sample_count = len(sensitivity)
    coefficient_report = coefficient_diagnostics(torch.cat(all_coefficients))
    gates = deployment_evidence_gates(
        identity_gain,
        statistics_gain,
        wrong_reference_gain,
        float(sensitivity.mean()),
        coefficient_report,
    )
    return {
        "metrics": summaries,
        "pairedImprovementOverIdentity": identity_gain,
        "pairedImprovementOverStatisticsMatch": statistics_gain,
        "pairedImprovementFromCorrectReference": wrong_reference_gain,
        "referenceSensitivityMae": {
            "mean": float(sensitivity.mean()),
            "minimum": float(sensitivity.min()),
            "maximum": float(sensitivity.max()),
            "sampleCount": sample_count,
        },
        "mixtureDiagnostics": coefficient_report,
        "runtime": {
            "singleModelAndLutMillisecondsPerImage": elapsed_seconds * 1000 / sample_count,
            "cudaPeakMemoryBytes": peak_memory,
            "parameterCount": sum(parameter.numel() for parameter in model.parameters()),
        },
        "deploymentEvidenceGates": gates,
        "minimumPracticalEffects": MINIMUM_PRACTICAL_EFFECTS,
        "allEvidenceGatesPass": all(gates.values()),
    }


def build_loader(dataset, batch_size: int, shuffle: bool, workers: int, use_cuda: bool):
    """Create a DataLoader tuned for either a local ZIP or Kaggle directories."""
    options = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": workers,
        "pin_memory": use_cuda,
        "persistent_workers": workers > 0,
    }
    if workers > 0:
        # Each worker prepares upcoming batches while the GPU handles the current
        # one. This helps extracted Kaggle files but can hurt random ZIP access.
        options["prefetch_factor"] = 2
    return DataLoader(**options)


def parse_arguments() -> argparse.Namespace:
    """Describe every reproducible choice accepted by the training command."""
    parser = argparse.ArgumentParser(
        description="Train ColorCheck's reference-conditioned 3D LUT predictor"
    )
    source = parser.add_mutually_exclusive_group(required=True)
    source.add_argument("--dataset-zip", type=Path, help="Local FiveK ZIP archive")
    source.add_argument(
        "--dataset-root",
        type=Path,
        help="Extracted FiveK directory or Kaggle mount containing raw/a/b/c/d/e",
    )
    parser.add_argument("--output-dir", type=Path, default=Path(".local/reference-matcher"))
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument(
        "--evaluation-image-size",
        type=int,
        default=256,
        help="Higher-resolution validation/test size; does not change ONNX input size",
    )
    parser.add_argument("--learning-rate", type=float, default=2e-4)
    parser.add_argument(
        "--workers",
        type=int,
        default=0,
        help="Use 0 for removable ZIP media and 2-4 for extracted Kaggle files",
    )
    parser.add_argument("--device", default="auto")
    parser.add_argument(
        "--mixed-precision",
        choices=("auto", "off", "fp16", "bf16"),
        default="auto",
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max-train-pairs", type=int, default=0)
    parser.add_argument("--max-validation-pairs", type=int, default=0)
    parser.add_argument("--max-test-pairs", type=int, default=0)
    parser.add_argument("--resume", type=Path)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument(
        "--synthetic-probability",
        type=float,
        default=0.5,
        help="Fraction of training pairs using exact synthetic style supervision",
    )
    parser.add_argument(
        "--ranking-weight",
        type=float,
        default=0.25,
        help="Weight for correct-reference versus wrong-style ranking",
    )
    parser.add_argument(
        "--exact-lut-weight",
        type=float,
        default=1.0,
        help="Weight for exact full-cube supervision on synthetic styles",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_arguments()
    dataset_path = args.dataset_zip or args.dataset_root

    # Seeding all three random-number libraries makes split creation, network
    # initialization, and augmentation choices repeatable where hardware allows.
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    scenes = inspect_dataset(dataset_path)
    splits = create_splits(scenes, seed=args.seed)
    splits.save(args.output_dir / "splits.json")

    train_data = FiveKReferenceDataset(
        dataset_path,
        splits.train,
        args.image_size,
        training=True,
        seed=args.seed,
        synthetic_probability=args.synthetic_probability,
        include_lut_supervision=True,
    )
    validation_data = FiveKReferenceDataset(
        dataset_path,
        splits.validation,
        args.evaluation_image_size,
        training=False,
        seed=args.seed,
    )
    synthetic_validation_data = FiveKReferenceDataset(
        dataset_path,
        splits.validation,
        args.evaluation_image_size,
        training=False,
        seed=args.seed,
        synthetic_probability=1.0,
    )
    test_data = FiveKReferenceDataset(
        dataset_path,
        splits.test,
        args.evaluation_image_size,
        training=False,
        seed=args.seed,
    )
    synthetic_test_data = FiveKReferenceDataset(
        dataset_path,
        splits.test,
        args.evaluation_image_size,
        training=False,
        seed=args.seed,
        synthetic_probability=1.0,
    )

    # Pair limits make a real but deliberately small smoke test possible before
    # spending GPU quota on the complete 20,000-pair training split.
    if args.max_train_pairs:
        train_data = Subset(train_data, range(min(args.max_train_pairs, len(train_data))))
    if args.max_validation_pairs:
        validation_data = Subset(
            validation_data, range(min(args.max_validation_pairs, len(validation_data)))
        )
        synthetic_validation_data = Subset(
            synthetic_validation_data,
            range(min(args.max_validation_pairs, len(synthetic_validation_data))),
        )
    if args.max_test_pairs:
        test_data = Subset(test_data, range(min(args.max_test_pairs, len(test_data))))
        synthetic_test_data = Subset(
            synthetic_test_data,
            range(min(args.max_test_pairs, len(synthetic_test_data))),
        )

    device = choose_device(args.device)
    mixed_precision = choose_mixed_precision(args.mixed_precision, device)
    use_cuda = device.type == "cuda"
    train_loader = build_loader(
        train_data, args.batch_size, shuffle=True, workers=args.workers, use_cuda=use_cuda
    )
    validation_loader = build_loader(
        validation_data,
        args.batch_size,
        shuffle=False,
        workers=args.workers,
        use_cuda=use_cuda,
    )
    synthetic_validation_loader = build_loader(
        synthetic_validation_data,
        args.batch_size,
        shuffle=False,
        workers=args.workers,
        use_cuda=use_cuda,
    )
    test_loader = build_loader(
        test_data, args.batch_size, shuffle=False, workers=args.workers, use_cuda=use_cuda
    )
    synthetic_test_loader = build_loader(
        synthetic_test_data,
        args.batch_size,
        shuffle=False,
        workers=args.workers,
        use_cuda=use_cuda,
    )

    model = ReferenceLutModel().to(device)
    loss_function = ReferenceMatchLoss().to(device)
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    # FP16 has a smaller numeric range, so GradScaler prevents small gradients
    # from becoming zero. BF16 and full precision do not require scaling.
    scaler = torch.amp.GradScaler(
        "cuda", enabled=(device.type == "cuda" and mixed_precision == torch.float16)
    )

    best_selection_score = float("inf")
    history = []
    starting_epoch = 1
    epochs_without_improvement = 0

    if args.resume:
        resume = torch.load(args.resume, map_location=device, weights_only=True)
        model.load_state_dict(resume["model"])
        optimizer.load_state_dict(resume["optimizer"])
        scheduler.load_state_dict(resume["scheduler"])
        if "scaler" in resume:
            scaler.load_state_dict(resume["scaler"])
        starting_epoch = resume["epoch"] + 1
        # Older checkpoints selected on real validation alone. Supporting their
        # field keeps interrupted smoke runs resumable without weakening new runs.
        best_selection_score = resume.get(
            "best_selection_score", resume.get("best_delta_e", float("inf"))
        )
        history_path = args.output_dir / "history.json"
        if history_path.exists():
            history = json.loads(history_path.read_text())

    setup = {
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if use_cuda else None,
        "mixedPrecision": str(mixed_precision).removeprefix("torch.") if mixed_precision else "off",
        "scenes": len(scenes),
        "trainPairs": len(train_data),
        "validationPairs": len(validation_data),
        "syntheticValidationPairs": len(synthetic_validation_data),
        "testPairs": len(test_data),
        "syntheticTrainingProbability": args.synthetic_probability,
        "referenceRankingWeight": args.ranking_weight,
        "exactLutWeight": args.exact_lut_weight,
        "workers": args.workers,
        "batchSize": args.batch_size,
        "trainingImageSize": args.image_size,
        "evaluationImageSize": args.evaluation_image_size,
    }
    print(json.dumps(setup))

    for epoch in range(starting_epoch, args.epochs + 1):
        model.train()
        running_loss = 0.0
        running_ranking_loss = 0.0
        running_exact_lut_loss = 0.0
        started = time.perf_counter()
        progress = tqdm(train_loader, desc=f"epoch {epoch}/{args.epochs}")

        for batch in progress:
            source, reference, expected = move_images(batch, device)
            wrong_reference, _ = mismatch_references(reference, batch["style_id"])
            # PyTorch accumulates gradients by default. Clearing them before
            # every batch ensures each update represents only the current batch.
            optimizer.zero_grad(set_to_none=True)

            with autocast_context(device, mixed_precision):
                # Correct and wrong references share one accelerator call. The
                # second half is a training control and is absent from ONNX.
                paired_source = torch.cat((source, source), dim=0)
                paired_reference = torch.cat((reference, wrong_reference), dim=0)
                paired_lut, _ = model(paired_source, paired_reference)
                lut, wrong_lut = paired_lut.chunk(2, dim=0)
                output = apply_lut(source, lut)
                wrong_output = apply_lut(source, wrong_lut)

            loss, _ = loss_function(
                output.float(), expected.float(), source.float(), lut.float()
            )
            ranking = reference_ranking_loss(
                output.float(), wrong_output.float(), expected.float()
            )
            exact_lut = exact_lut_supervision_loss(
                lut,
                batch["expected_lut"].to(device, non_blocking=use_cuda),
                batch["has_exact_lut"].to(device, non_blocking=use_cuda),
            )
            loss = (
                loss
                + args.ranking_weight * ranking
                + args.exact_lut_weight * exact_lut
            )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Gradient clipping limits rare unstable updates without changing
            # ordinary gradients below this norm.
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * source.shape[0]
            running_ranking_loss += ranking.item() * source.shape[0]
            running_exact_lut_loss += exact_lut.item() * source.shape[0]
            progress.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        real_validation = validate(
            model,
            validation_loader,
            device,
            loss_function,
            mixed_precision=mixed_precision,
        )
        synthetic_validation = validate(
            model,
            synthetic_validation_loader,
            device,
            loss_function,
            mixed_precision=mixed_precision,
        )
        selection_score = checkpoint_selection_score(
            real_validation, synthetic_validation
        )
        record = {
            "epoch": epoch,
            "trainLoss": running_loss / len(train_data),
            "referenceRankingLoss": running_ranking_loss / len(train_data),
            "exactLutLoss": running_exact_lut_loss / len(train_data),
            "seconds": time.perf_counter() - started,
            "learningRate": scheduler.get_last_lr()[0],
            "realValidation": real_validation,
            "syntheticValidation": synthetic_validation,
            "selectionScore": selection_score,
        }
        history.append(record)
        (args.output_dir / "history.json").write_text(json.dumps(history, indent=2) + "\n")

        checkpoint = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "best_selection_score": min(best_selection_score, selection_score),
            "config": {
                "basis_count": model.basis_count,
                "lut_size": model.lut_size,
                "image_size": args.image_size,
                "seed": args.seed,
                "synthetic_probability": args.synthetic_probability,
                "ranking_weight": args.ranking_weight,
                "exact_lut_weight": args.exact_lut_weight,
            },
            # validation remains an alias for compatibility with the ONNX
            # manifest and earlier experiment readers.
            "validation": real_validation,
            "synthetic_validation": synthetic_validation,
            "selection_score": selection_score,
        }
        # ``last.pt`` supports recovery from interruption. ``best.pt`` remains
        # fixed at the strongest validation result even if later epochs regress.
        torch.save(checkpoint, args.output_dir / "last.pt")
        if selection_score < best_selection_score:
            best_selection_score = selection_score
            epochs_without_improvement = 0
            torch.save(checkpoint, args.output_dir / "best.pt")
        else:
            epochs_without_improvement += 1

        print(json.dumps(record))
        if epochs_without_improvement >= args.patience:
            print(f"Early stopping after {args.patience} validation epochs without improvement")
            break

    # Final testing happens exactly once with the strongest validation checkpoint.
    # Test scores therefore cannot influence which checkpoint is selected.
    best_checkpoint = torch.load(args.output_dir / "best.pt", map_location=device, weights_only=True)
    model.load_state_dict(best_checkpoint["model"])
    test_metrics = validate(
        model, test_loader, device, loss_function, mixed_precision=mixed_precision
    )
    shuffle_metrics = validate(
        model,
        test_loader,
        device,
        loss_function,
        mixed_precision=mixed_precision,
        shuffled_reference=True,
    )
    baseline_metrics = identity_baseline(test_loader, device)
    real_evaluation = comprehensive_evaluation(
        model, test_loader, device, mixed_precision, seed=args.seed
    )
    synthetic_evaluation = comprehensive_evaluation(
        model, synthetic_test_loader, device, mixed_precision, seed=args.seed + 1
    )
    summary = {
        "bestEpoch": best_checkpoint["epoch"],
        "bestValidation": best_checkpoint["validation"],
        "bestSyntheticValidation": best_checkpoint["synthetic_validation"],
        "bestSelectionScore": best_checkpoint["selection_score"],
        "test": test_metrics,
        "testWithShuffledReferences": shuffle_metrics,
        "identityTestBaseline": baseline_metrics,
        "realExpertEvaluation": real_evaluation,
        "syntheticReferenceEvaluation": synthetic_evaluation,
        "allDeploymentEvidenceGatesPass": (
            real_evaluation["allEvidenceGatesPass"]
            and synthetic_evaluation["allEvidenceGatesPass"]
        ),
    }
    (args.output_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


### `render_comparisons.py`


In [ ]:
%%writefile /kaggle/working/tools/ml/reference_matcher/render_comparisons.py
"""Render a held-out comparison sheet for qualitative photographic review.

Metrics decide whether a model is eligible for deployment, but a compact contact
sheet helps a person spot objectionable casts, flattened skin, clipped skies, or
other failures that one aggregate number can hide. The sheet is an experiment
artifact only; dataset photographs are never committed to the repository.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

from .dataset import FiveKReferenceDataset, SplitManifest
from .evaluation import statistics_match
from .model import ReferenceLutModel, apply_lut


def load_splits(path: Path) -> SplitManifest:
    """Reconstruct the immutable scene split recorded by training."""
    payload = json.loads(path.read_text())
    return SplitManifest(
        train=tuple(payload["train"]),
        validation=tuple(payload["validation"]),
        test=tuple(payload["test"]),
    )


def tensor_to_image(tensor: torch.Tensor, tile_size: int) -> Image.Image:
    """Convert one bounded CHW tensor into a consistently sized RGB tile."""
    pixels = (
        tensor.detach().float().clamp(0, 1).mul(255).round().byte().permute(1, 2, 0)
    )
    image = Image.fromarray(pixels.cpu().numpy())
    return image.resize((tile_size, tile_size), Image.Resampling.LANCZOS)


def sample_indices(dataset_size: int, count: int) -> list[int]:
    """Choose reproducible, evenly distributed examples across scenes/experts."""
    if dataset_size < 1 or count < 1:
        raise ValueError("Dataset size and sample count must both be positive")
    count = min(count, dataset_size)
    if count == 1:
        return [0]
    return [round(index * (dataset_size - 1) / (count - 1)) for index in range(count)]


@torch.no_grad()
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--splits", type=Path, required=True)
    source = parser.add_mutually_exclusive_group(required=True)
    source.add_argument("--dataset-zip", type=Path)
    source.add_argument("--dataset-root", type=Path)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--samples", type=int, default=10)
    parser.add_argument("--tile-size", type=int, default=192)
    args = parser.parse_args()

    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    config = checkpoint["config"]
    model = ReferenceLutModel(config["basis_count"], config["lut_size"])
    model.load_state_dict(checkpoint["model"])
    model.eval()

    splits = load_splits(args.splits)
    dataset = FiveKReferenceDataset(
        args.dataset_zip or args.dataset_root,
        splits.test,
        image_size=config["image_size"],
        training=False,
        seed=config["seed"],
    )

    labels = ("Source", "Reference", "Expected", "Learned", "Statistics")
    header_height = 30
    row_height = args.tile_size + header_height
    row_count = min(args.samples, len(dataset))
    sheet = Image.new(
        "RGB",
        (len(labels) * args.tile_size, header_height + row_height * row_count),
        "white",
    )
    draw = ImageDraw.Draw(sheet)
    font = ImageFont.load_default()
    for column, label in enumerate(labels):
        draw.text((column * args.tile_size + 8, 10), label, fill="black", font=font)

    for row, index in enumerate(sample_indices(len(dataset), args.samples)):
        sample = dataset[index]
        source_image = sample["source"].unsqueeze(0)
        reference = sample["reference"].unsqueeze(0)
        lut, _ = model(source_image, reference)
        learned = apply_lut(source_image, lut)[0]
        baseline = statistics_match(source_image, reference)[0]
        tiles = (
            sample["source"],
            sample["reference"],
            sample["expected"],
            learned,
            baseline,
        )
        row_top = header_height + row * row_height
        draw.text(
            (8, row_top + 9),
            f"{sample['scene']} | expert {sample['expert']}",
            fill="black",
            font=font,
        )
        for column, tile in enumerate(tiles):
            sheet.paste(
                tensor_to_image(tile, args.tile_size),
                (column * args.tile_size, row_top + header_height),
            )

    args.output.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(args.output, format="JPEG", quality=92, optimize=True)
    print(f"Wrote held-out comparison sheet: {args.output}")


if __name__ == "__main__":
    main()


## 3. Run the smoke experiment

This run uses real FiveK images and performs genuine gradient updates, but its small sample count is only a systems test. It is not evidence of product-quality improvement.

In [ ]:
smoke_command = [
    sys.executable, '-m', 'tools.ml.reference_matcher.train',
    '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_ROOT / 'smoke'),
    '--epochs', '1',
    '--batch-size', '32',
    '--image-size', '128',
    '--evaluation-image-size', '256',
    '--workers', '2',
    '--mixed-precision', 'auto',
    '--max-train-pairs', '256',
    '--max-validation-pairs', '64',
    '--max-test-pairs', '64',
]
subprocess.run(smoke_command, cwd='/kaggle/working', check=True)


## 4. Review the measurements

The identity baseline asks whether training improved upon leaving the source untouched. The shuffled-reference control asks whether the model actually responds to its reference rather than applying one generic look.

In [ ]:
import json

summary_path = OUTPUT_ROOT / 'smoke' / 'summary.json'
history_path = OUTPUT_ROOT / 'smoke' / 'history.json'
print(json.dumps(json.loads(summary_path.read_text()), indent=2))
print(json.dumps(json.loads(history_path.read_text()), indent=2))


In [ ]:
comparison_command = [
    sys.executable, '-m', 'tools.ml.reference_matcher.render_comparisons',
    '--dataset-root', str(DATASET_ROOT),
    '--checkpoint', str(OUTPUT_ROOT / 'smoke' / 'best.pt'),
    '--splits', str(OUTPUT_ROOT / 'smoke' / 'splits.json'),
    '--output', str(OUTPUT_ROOT / 'smoke' / 'held-out-comparisons.jpg'),
    '--samples', '10',
]
subprocess.run(comparison_command, cwd='/kaggle/working', check=True)


## 5. Measured medium pilot

The validated smoke run measured the complete systems path. This pilot uses 5,000 training pairs over three epochs and 500-pair validation/test controls, remaining well inside one GPU session. The best checkpoint remains separate from the latest.

In [ ]:
RUN_MEDIUM_PILOT = True

if RUN_MEDIUM_PILOT:
    pilot_command = [
        sys.executable, '-m', 'tools.ml.reference_matcher.train',
        '--dataset-root', str(DATASET_ROOT),
        '--output-dir', str(OUTPUT_ROOT / 'pilot'),
        '--epochs', '3',
        '--batch-size', '32',
        '--image-size', '128',
        '--evaluation-image-size', '256',
        '--workers', '4',
        '--mixed-precision', 'auto',
        '--patience', '2',
        '--max-train-pairs', '5000',
        '--max-validation-pairs', '500',
        '--max-test-pairs', '500',
    ]
    subprocess.run(pilot_command, cwd='/kaggle/working', check=True)
else:
    print('Medium pilot is disabled.')
